In [7]:
import simpy
import numpy as np
import random

In [4]:
def example(env):
    event = simpy.events.Timeout(env, delay=1, value=42)
    value = yield event
    print('now=%d, value=%d' % (env.now, value))

env = simpy.Environment()
example_gen = example(env)
p = simpy.events.Process(env, example_gen)

env.run()

now=1, value=42


In [6]:
# first try, simulate one bus stop 
# bus arrives every 10 mins, capacity 50
# passengers arrive every 3 mins, 1-4 passengers

def bus_stop(env):
    time_interval = 1
    bus_interval = 10
    passenger_interval = 3
    bus_capacity = 50
    passengers_waiting = 0
    t = 0

    while True:
        # simulate time
        yield env.timeout(time_interval)
        t = t+1

        if t % passenger_interval ==0:
            passengers_arriving = np.random.randint(1,5)
            passengers_waiting += passengers_arriving
            print(f"Time {t}: {passengers_arriving} passengers arrived at the bus stop.")

        if t % bus_interval == 0:
            bus_passengers = np.random.randint(0, bus_capacity+1)
            bus_remaining_capacity = bus_capacity - bus_passengers
            passengers_boarding = min(passengers_waiting, bus_remaining_capacity)
            passengers_waiting -= passengers_boarding
            print(f"Time {t}: Bus arrives with {bus_passengers} passengers. Remaining capacity: {bus_remaining_capacity}")
            print(f"Time {t}: {passengers_boarding} passengers boarded the bus. Passengers waiting: {passengers_waiting}")

env = simpy.Environment()
env.process(bus_stop(env))
env.run(until=60)  # Run the simulation for 60 time units

Time 3: 1 passengers arrived at the bus stop.
Time 6: 4 passengers arrived at the bus stop.
Time 9: 3 passengers arrived at the bus stop.
Time 10: Bus arrives with 45 passengers. Remaining capacity: 5
Time 10: 5 passengers boarded the bus. Passengers waiting: 3
Time 12: 2 passengers arrived at the bus stop.
Time 15: 1 passengers arrived at the bus stop.
Time 18: 2 passengers arrived at the bus stop.
Time 20: Bus arrives with 29 passengers. Remaining capacity: 21
Time 20: 8 passengers boarded the bus. Passengers waiting: 0
Time 21: 3 passengers arrived at the bus stop.
Time 24: 2 passengers arrived at the bus stop.
Time 27: 4 passengers arrived at the bus stop.
Time 30: 2 passengers arrived at the bus stop.
Time 30: Bus arrives with 35 passengers. Remaining capacity: 15
Time 30: 11 passengers boarded the bus. Passengers waiting: 0
Time 33: 2 passengers arrived at the bus stop.
Time 36: 2 passengers arrived at the bus stop.
Time 39: 2 passengers arrived at the bus stop.
Time 40: Bus arri

In [8]:
def passenger_generator(env, bus_stop):
    """Generates passengers arriving at the stop."""
    passenger_id = 0
    while True:
        # Wait 3 minutes before the next passenger group arrives
        yield env.timeout(3)
        
        # 1 to 4 passengers arrive
        num_arriving = random.randint(1, 4)
        for _ in range(num_arriving):
            passenger_id += 1
            # Put the passenger into the 'bus_stop' store
            yield bus_stop.put(passenger_id) 
        
        print(f"Time {env.now:>3}: {num_arriving} passengers arrived. Total waiting: {len(bus_stop.items)}")

def bus_arrival(env, bus_stop):
    """Buses arrive every 10 minutes."""
    while True:
        yield env.timeout(10)
        
        # Bus arrives with a random number of passengers already on board
        current_passengers = random.randint(0, 10) # Assuming it's mostly empty
        remaining_capacity = 50 - current_passengers
        
        boarded = 0
        # While the bus has space AND there are people waiting
        while remaining_capacity > 0 and len(bus_stop.items) > 0:
            # Get the next passenger from the waiting room
            passenger = yield bus_stop.get()
            boarded += 1
            remaining_capacity -= 1
            
        print(f"Time {env.now:>3}: Bus arrived. Boarded {boarded} people. Left waiting: {len(bus_stop.items)}")

# Setup
env = simpy.Environment()
# Create a waiting room that holds infinite passengers
bus_stop = simpy.Store(env) 

env.process(passenger_generator(env, bus_stop))
env.process(bus_arrival(env, bus_stop))

env.run(until=60) # Run for 30 minutes

Time   3: 2 passengers arrived. Total waiting: 2
Time   6: 2 passengers arrived. Total waiting: 4
Time   9: 1 passengers arrived. Total waiting: 5
Time  10: Bus arrived. Boarded 5 people. Left waiting: 0
Time  12: 1 passengers arrived. Total waiting: 1
Time  15: 1 passengers arrived. Total waiting: 2
Time  18: 3 passengers arrived. Total waiting: 5
Time  20: Bus arrived. Boarded 5 people. Left waiting: 0
Time  21: 2 passengers arrived. Total waiting: 2
Time  24: 1 passengers arrived. Total waiting: 3
Time  27: 2 passengers arrived. Total waiting: 5
Time  30: 2 passengers arrived. Total waiting: 4
Time  30: Bus arrived. Boarded 7 people. Left waiting: 0
Time  33: 4 passengers arrived. Total waiting: 4
Time  36: 2 passengers arrived. Total waiting: 6
Time  39: 1 passengers arrived. Total waiting: 7
Time  40: Bus arrived. Boarded 7 people. Left waiting: 0
Time  42: 3 passengers arrived. Total waiting: 3
Time  45: 2 passengers arrived. Total waiting: 5
Time  48: 4 passengers arrived. Total

In [56]:
class Passenger:
    def __init__(self, env, name):
        self.env = env
        self.name = name
        self.arrival_time = env.now 

#class Bus:
#    def __init__(self, env, capacity):
#        self.env = env
#        self.capacity = capacity
#        self.passengers_on_board = min(round(random.expovariate(1.0 / 15.0)), capacity)


def passenger_generator(env, bus_stop):
    """Generates passengers arriving at the stop."""
    passenger_id = 0
    while True:
        # Average time between arrivals is 3 minutes
        time_to_next_arrival = random.expovariate(1.0 / 3.0) 
        yield env.timeout(time_to_next_arrival)


        # 1 to 4 passengers arrive
        num_arriving = random.randint(1, 4)
        env.total_passengers_generated += num_arriving
        for _ in range(num_arriving):
            passenger_id += 1
            p = Passenger(env, passenger_id)
            # Put the passenger into the 'bus_stop' store
            yield bus_stop.put(p) 

        
        #print(f"Time {env.now:>3}: {num_arriving} passengers arrived. Total waiting: {len(bus_stop.items)}")

def bus_arrival(env, bus_stop):
    """Buses arrive every 10 minutes."""

    
    while True:
        time_to_next_bus = random.expovariate(1.0 / 10.0)
        yield env.timeout(time_to_next_bus)
        env.bus_wait_times.append(time_to_next_bus)
        
        # Bus arrives with a random number of passengers already on board
        current_passengers = min(round(random.expovariate(1.0 / 7.0)), 20)
        remaining_capacity = 50 - current_passengers
        
        boarded = 0
        # While the bus has space AND there are people waiting
        while remaining_capacity > 0 and len(bus_stop.items) > 0:
            # Get the next passenger from the waiting room
            p = yield bus_stop.get()
            env.passenger_wait_times.append(env.now - p.arrival_time)
            boarded += 1
            env.total_passengers_boarded += 1
            remaining_capacity -= 1
            
        #print(f"Time {env.now:>3}: Bus arrived with {current_passengers} passengers. Boarded {boarded} people. Left waiting: {len(bus_stop.items)}")

# Setup
env = simpy.Environment()
# Create a waiting room that holds infinite passengers
bus_stop = simpy.Store(env) 
env.passenger_wait_times = []
env.bus_wait_times = []
env.total_passengers_generated = 0
env.total_passengers_boarded = 0



env.process(passenger_generator(env, bus_stop))
env.process(bus_arrival(env, bus_stop))

env.run(until=60)
print(f"Average passenger wait time: {np.mean(env.passenger_wait_times):.2f} minutes")
print(f"Average bus wait time: {np.mean(env.bus_wait_times):.2f} minutes")
print(f"Total passengers generated: {env.total_passengers_generated}")
print(f"Total passengers boarded: {env.total_passengers_boarded}")
print(f"Passengers left waiting: {len(bus_stop.items)}")

Average passenger wait time: 6.34 minutes
Average bus wait time: 9.25 minutes
Total passengers generated: 43
Total passengers boarded: 42
Passengers left waiting: 1
